# Phase 2 Bounded Projected-Depth Semantic Occupancy IoU

This notebook runs a bounded streamed CARLA validation evaluation and inspects the artifacts produced by `src.evaluation.phase2_carla_eval`.

Important: these are **bounded projected-depth semantic occupancy IoU** results. The target proxy is built from LiDAR-converted projected depth plus front semantic segmentation, not dense simulator 3D occupancy ground truth.

## 1. Setup

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

from IPython.display import Image, Markdown, display

from src.evaluation.phase2_carla_eval import (
    Phase2CarlaEvalConfig,
    iter_shadow_records,
    run_bounded_carla_iou_evaluation,
)

MAX_FRAMES = 200
OUTPUT_DIR = Path("outputs/phase2")
OCCUPANCY_THRESHOLD = 0.25
DISAGREEMENT_THRESHOLD = 0.05

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Run Bounded Evaluation

In [ ]:
config = Phase2CarlaEvalConfig(
    max_frames=MAX_FRAMES,
    output_dir=OUTPUT_DIR,
    occupancy_threshold=OCCUPANCY_THRESHOLD,
    disagreement_threshold=DISAGREEMENT_THRESHOLD,
    bev_frame_count=5,
)

artifacts = run_bounded_carla_iou_evaluation(config=config)
for name, path in artifacts.__dict__.items():
    print(f"{name}: {path}")

## 3. IoU Table

In [ ]:
baseline = json.loads(artifacts.baseline_iou.read_text(encoding="utf-8"))
temporal = json.loads(artifacts.temporal_iou.read_text(encoding="utf-8"))
run_summary = json.loads(artifacts.run_summary.read_text(encoding="utf-8"))

baseline_by_class = {row["class_id"]: row for row in baseline["classes"]}
temporal_by_class = {row["class_id"]: row for row in temporal["classes"]}

lines = ["| class_name | baseline_iou | temporal_iou | delta |", "|---|---:|---:|---:|"]
for class_id, base_row in baseline_by_class.items():
    temp_row = temporal_by_class[class_id]
    delta = temp_row["iou"] - base_row["iou"]
    lines.append(
        f"| {base_row['class_name']} | {base_row['iou']:.3f} | "
        f"{temp_row['iou']:.3f} | {delta:+.3f} |"
    )

display(Markdown("\n".join(lines)))
display(Markdown(f"**Processed frames:** {run_summary['processed_frame_count']} / requested {run_summary['requested_max_frames']}"))
display(Markdown(f"**Target proxy:** {run_summary['target_proxy']}"))
display(Markdown(f"**Free-space note:** {run_summary['free_space_note']}"))

## 4. Shadow-Mode Summary

In [ ]:
records = list(iter_shadow_records(artifacts.shadow_records))
clusters = json.loads(artifacts.shadow_clusters.read_text(encoding="utf-8"))
rates = [record["disagreement_rate"] for record in records]
flagged = [record for record in records if record["flagged"]]

summary_lines = [
    f"- Evaluated frames: `{len(records)}`",
    f"- Flagged frames: `{len(flagged)}`",
    f"- Mean disagreement: `{(sum(rates) / len(rates)) if rates else 0.0:.4f}`",
    f"- Max disagreement: `{max(rates) if rates else 0.0:.4f}`",
    f"- Disagreement threshold: `{clusters['disagreement_threshold']}`",
]
display(Markdown("\n".join(summary_lines)))
display(Markdown("```json\n" + json.dumps(clusters["clusters"], indent=2) + "\n```"))

## 5. Visual Checks

In [ ]:
for image_path in artifacts.bev_images[:5]:
    display(Markdown(f"**{image_path.name}**"))
    display(Image(filename=str(image_path)))

## 6. Interpretation Notes

In [ ]:
notes = []
for class_id, base_row in baseline_by_class.items():
    temp_row = temporal_by_class[class_id]
    delta = temp_row["iou"] - base_row["iou"]
    if delta >= 0:
        notes.append(f"- `{base_row['class_name']}` temporal IoU preserved or improved by `{delta:+.3f}`.")
    else:
        notes.append(f"- `{base_row['class_name']}` temporal IoU dropped by `{delta:+.3f}`; inspect BEV outputs for stale fused occupancy or sparse-depth artifacts.")

notes.extend([
    "- Vehicle and pedestrian IoU may be noisy because the depth source is sparse 32-channel LiDAR projected into the front camera.",
    "- Shadow-mode clusters are failure-mode hints, not final causal claims.",
    "- Baseline IoU is expected to be high because the proxy target is built from the current frame's own projected depth and segmentation.",
])
display(Markdown("\n".join(notes)))